# 01 - Data Preprocessing

This notebook prepares the DUTS dataset. It uses normal Python code for checking files and reading the manifest, and uses `subprocess` only to run the existing `pre_processing.py` script.

## Step 1 - Setup

In [ ]:
from pathlib import Path
import json
import sys

def find_notebooks_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd / "salient-object-detection-cnn" / "notebooks"]
    for parent in cwd.parents:
        candidates.extend([parent / "notebooks", parent / "salient-object-detection-cnn" / "notebooks"])
    for candidate in candidates:
        if (candidate / "notebook_utils.py").exists():
            return candidate
    raise FileNotFoundError("Could not find notebooks/notebook_utils.py")

NOTEBOOKS_DIR = find_notebooks_dir()

sys.path.insert(0, str(NOTEBOOKS_DIR))

from notebook_utils import PROJECT_DIR, run_script

PROJECT_DATA_DIR = PROJECT_DIR / "data"
if not PROJECT_DATA_DIR.exists():
    PROJECT_DATA_DIR = (PROJECT_DIR / "../data").resolve()

sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

## Step 2 - Verify Raw DUTS Folders

In [ ]:
raw_folders = {
    "train_images": PROJECT_DATA_DIR / "DUTS-TR/DUTS-TR-Image",
    "train_masks": PROJECT_DATA_DIR / "DUTS-TR/DUTS-TR-Mask",
    "test_images": PROJECT_DATA_DIR / "DUTS-TE/DUTS-TE-Image",
    "test_masks": PROJECT_DATA_DIR / "DUTS-TE/DUTS-TE-Mask",
}

for name, folder in raw_folders.items():
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")
    files = [path for path in folder.iterdir() if path.is_file()]
    print(f"{name}: {len(files)} files")

## Step 3 - Run Preprocessing Script

In [ ]:
IMAGE_SIZE = 128
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42

run_script(
    "pre_processing.py",
    "--raw_data_dir", PROJECT_DATA_DIR,
    "--output_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--train_split", TRAIN_SPLIT,
    "--val_split", VAL_SPLIT,
    "--test_split", TEST_SPLIT,
    "--seed", SEED,
)

## Step 4 - Inspect Generated Manifest

In [ ]:
manifest_path = PROJECT_DIR / "pre-processed/manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("pre-processed/manifest.json was not created.")

manifest = json.loads(manifest_path.read_text())
print("Image size:", manifest["image_size"])
print("Train split:", manifest["train_split"])
print("Validation split:", manifest["val_split"])
print("Test split:", manifest["test_split"])
print("Seed:", manifest["seed"])
print("Counts:", manifest["counts"])